<a href="https://colab.research.google.com/github/NoaTal1996/Backbone-Graph/blob/main/code/Topic_Clustring_and_NER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [ ]:
!pip install bertopic
!pip install tweet-preprocessor

In [ ]:
from google.colab import drive

import pandas as pd
import matplotlib.pyplot as plt

import re

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from pysentimiento.preprocessing import preprocess_tweet

# Load Data

## Connect to Goole Drive


In [ ]:
# Connect to Goole Drive to use the data.
drive.mount('/content/drive')

## Load Labeled Datasets for Research on Information Operations.

In [ ]:
file_path = "/content/drive/Shareddrives/KeyBoostAuthor/Labeled_Datasets/Ecuador/Ecuador_part_3.gzip.parquet"

df = pd.read_parquet(file_path)

In [ ]:
df.head()

## Dataset Statistics

In [ ]:
total_posts = len(df)
hashtag_posts = df[df['hashtags'].notna() & (df['hashtags'].str.len() > 0)]
mention_posts = df[df['account_mentions'].notna() & (df['account_mentions'].str.len() > 0)]
url_posts = df[df['urls'].notna() & (df['urls'].str.len() > 0)]

print(f"{'df shape:':<40} {df.shape}")
print(f"{'df posts with hashtags # shape:':<40} {hashtag_posts.shape}  ({len(hashtag_posts)/total_posts*100:.2f}%)")
print(f"{'df posts with account_mentions @ shape:':<40} {mention_posts.shape}  ({len(mention_posts)/total_posts*100:.2f}%)")
print(f"{'df posts with urls shape:':<40} {url_posts.shape}  ({len(url_posts)/total_posts*100:.2f}%)")


print()
print()

print(f"{'**collum name**':<30} **value type**")
for col in df.columns:
    dtype_str = str(df[col].dtype)  # Convert dtype to string
    # get a sample non-null value if exists
    sample_val = df[col].dropna().iloc[0] if df[col].notna().any() else None
    print(f"{col:<30} | {type(sample_val).__name__}")


print()
print()
print(df['post_language'].value_counts().head())

# Clean Text

In [ ]:
def clean_url(url_match):

    url = url_match.group(0)

    # 1. Remove protocol (http:// or https://)
    url = re.sub(r"^https?://", "", url)

    # 2. Remove query parameters (?x=1)
    url = url.split("?")[0]

    # 3. Split into components by "/"
    parts = url.split("/")

    # Domain (first part)
    domain = parts[0].replace(".", "_")  # twitter.com → twitter_com

    # First section (second part of the URL)
    if len(parts) > 1 and parts[1].strip() != "":
        section = parts[1].replace(".", "_")
        token = f" URL_{domain}_{section} "
    else:
        token = f" URL_{domain} "

    return token


In [ ]:
def clean_text(text: str) -> str:

    # lowercase
    text = str(text).lower()

    # URLs → domian + first section
    text = re.sub(r"https?://\S+|www\.\S+", clean_url, text)

    # Hashtags → HASHTAG_xxx
    text = re.sub(r"#(\w+)", r" HASHTAG_\1 ", text)

    # Mentions → MENTION_xxx
    text = re.sub(r"@(\w+)", r" MENTION_\1 ", text)

    # Remove non english-letters/numbers/underscore
    text = re.sub(r"[^0-9a-zA-Z_]+", " ", text)

    # Collapse spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [ ]:
df["clean_post_text"] = df["post_text"].apply(clean_text)

## Remove Engllish Stopwords

In [ ]:
from nltk.corpus import stopwords
import nltk

In [ ]:
nltk.download('stopwords')

EXTRA_STOPWORDS = [
    'he', 'she', 'they', 'you', 'i', 'we', 'im', 'hes', 'shes', 'its', 'rt', 'amp', 're',
    'theyre', 'dont', 'doesnt', 'would', 'could', 'like', 'know', 'say', 'said', 'one',
    'look', 'looks', 'think', 'oh', 'yeah', 'guy', 'gonna', 'thing', 'someone', 'people',
    'get', 'got', 'thats', 'just', 'really', 'even', 'still', 'want', 'need', 'needs',
    'me', 'to', 'you', 'the', 'that', 'be', 'my', 'get', 'in', 'and',
     'rt', 'the', 'https', 'to', 'my', 'it', 'on', 'you', 'of', 'for', 'https', 'http'
]
all_stopwords_english = set(stopwords.words('english') + EXTRA_STOPWORDS)

def remove_stopwords(text):
    return " ".join([w for w in text.split() if w not in all_stopwords_english])


In [ ]:
df["clean_post_text"] = df["post_text"].apply(remove_stopwords)

# English Topic Clustering

In [ ]:
def english_topic_modeling(posts_text_lst, min_topic_size=20):

    embedding_model = SentenceTransformer("digio/Twitter4SSE")

    topic_model = BERTopic(
        embedding_model=embedding_model,
        language="english",
        min_topic_size=min_topic_size,
        verbose=True
    )

    topics, probs = topic_model.fit_transform(posts_text_lst)

    return topics, probs, topic_model

In [ ]:
english_df = df[df["post_language"] == "en"].head(200).copy()
print(f"english_df shape: {english_df.shape}")

posts_text_lst = english_df["clean_post_text"].to_list()

topics, probs, topic_model = english_topic_modeling(posts_text_lst)

english_df["topic"] = topics

In [ ]:
topic_count = topic_model.get_topic_info()
topic_count

In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(topic_count.Topic, topic_count.Count)
plt.xlabel("Topic ID")
plt.ylabel("Number of Posts")
plt.title("Post Count per Topic ID")
plt.grid(axis='y')
plt.xticks(topic_count.Topic)
plt.tight_layout()
plt.show()

In [ ]:
topic_model.visualize_topics(custom_labels=True)

In [ ]:
topic_model.visualize_hierarchy()

